# Packages & Includes

This notebook was tested with Julia v.1.10 using a single core, with the environment defined in this folder. (see the .toml files).  

Since Julia v.1.12 the default Jupyterlab kernel uses multiple threads, which is known to lead to issues with PyPlot & PyCall.  
In particular, it might not even be possible to load these two packages.   
If you plan to use multi-threading or the latest version of Julia, you might need to find an alternative for the PyPlot plotting routines used here.   
The model itself, however has only lightweight, Julia-native dependencies and should not be affected.

In [ ]:
import Pkg; Pkg.activate(".")

In [ ]:
# Only needs to be done the first time to download dependencies
Pkg.instantiate()

In [ ]:
# Package dependencies
using FRANZ
using Healpix
using StatsBase, LinearAlgebra

# Plotting
using PyPlot, ColorSchemes, Colors
using PyCall
ll = pyimport("labellines")

A lot of the wrapper-functions used for plotting and model evaluation are defined in the files in "plotting_utility".   
It may be worthwhile to take a look.   

In [ ]:
# load utility functions
include("plotting_utility/main.jl")
set_plot_style() # this overwrites some of matplotlibs rc parameters to ensure that the plots render correctly.

# Model Definitions

Here the model parameters are described:  

###### Parameters describing the explosion(s)
E_51 (Real): Explosion energy in units of 10^51 erg.  
M_ej (Real): Ejecta Mass in Msol.   
f_CR (Real): Fraction of Explosion Energy converted into non-thermal (Cosmic Ray) Energy. 
α_p  (Real): Momentum boost factor after shell formation. (See Lancaster et al. 2024) 

###### Additional physics parameters
Λ_22 (Real OR function of density and temperature): Cooling rate/function at 10^6 K in units of 10^-22 erg/s cm^3.

###### Multiple explosions only: 
Δt_6 (Real): Time between SN Explosions in Myr. 

###### Parameters describing the environment of the explosion
n_0 (Real): sets scale of ambient density in units of amu/cm^3 (functional form is described in the environment).

## Fiducial Environment & Model parameters

In [ ]:
# Explosion Models
E_51 = 1
M_ej = 5
f_CR = 0
α_p  = 4.0
Λ_22 = 1.0
n_0 = 1.0

# Superbubble & Starburst parameters
Δt_SB  = 1
Δt_StB = 1e-4

# functions for continuous & bursty injection
SN_continuous(Δt_SN::Real) = (t::Real) -> 1.0 + t / Δt_SN
SN_bursty(Δt_SN::Real) = (t::Real) -> 1.0 + fld(t / Δt_SN, 1.0)

# Number of SN events for different models
N_SN_SB_continuous  = SN_continuous(Δt_SB)
N_SN_SB_bursty      = SN_bursty(Δt_SB)
N_SN_StB_continuous = SN_continuous(Δt_StB)
N_SN_StB_bursty     = SN_bursty(Δt_StB)

Explosion models can be defined as below, by calling Model(E_SN=..., M_ej=..., f_CR=..., α_p=..., Λ_cool=..., n_0=...) with appropriate arguments.   
The default is a single SN (1e51 erg) with 1 solar mass of ejecta and no momentum boost, CRs or cooling in an ambient medium with 1 amu/cc.

In [ ]:
# specify model parameters (E_SN, M_ej, f_CR, α_p, Λ_cool, n_0)
# Fiducial models
SN       = Model(E_51=E_51, M_ej=M_ej, f_CR=f_CR, α_p=α_p, Λ_22=Λ_22)
SB_cont  = Model(E_51=E_51 * N_SN_SB_continuous, M_ej=M_ej * N_SN_SB_continuous, f_CR=f_CR, α_p=α_p, Λ_22=Λ_22)
StB_cont = Model(E_51=E_51 * N_SN_StB_continuous, M_ej=M_ej * N_SN_StB_continuous, f_CR=f_CR, α_p=α_p, Λ_22=Λ_22)

# bursty models
SB_bursty  = Model(E_51=E_51 * N_SN_SB_bursty, M_ej=M_ej * N_SN_SB_bursty, f_CR=f_CR, α_p=α_p, Λ_22=Λ_22)
StB_bursty = Model(E_51=E_51 * N_SN_StB_bursty, M_ej=M_ej * N_SN_StB_bursty, f_CR=f_CR, α_p=α_p, Λ_22=Λ_22)

# (continuous) Models without cooling
SN_no_cool  = Model(E_51=E_51, M_ej=M_ej, f_CR=f_CR, α_p=α_p, Λ_22=0.0)
SB_no_cool  = Model(E_51=E_51 * N_SN_SB_continuous, M_ej=M_ej * N_SN_SB_continuous, f_CR=f_CR, α_p=α_p, Λ_22=0.0)
StB_no_cool = Model(E_51=E_51 * N_SN_StB_continuous, M_ej=M_ej * N_SN_StB_continuous, f_CR=f_CR, α_p=α_p, Λ_22=0.0)

## Characteristic scales

These are a number of analytical results from the literature/ limits derived from the model that are shown in some of the plots below.

In [ ]:
######## Single explosion
## Ejecta Dominated Phase
t_ED = 1.9e-4 * (M_ej)^(5/6) / sqrt(E_51) * n_0^(-1/3)
v_ED = sqrt(2 * 1e51 / Msol) / km_s * sqrt(E_51 / M_ej)
R_ED = (3 / 4π * Msol * M_ej / μ / n_0)^(1/3) / pc
p_ED = M_ej / 4π * v_ED 
E_ED = 1e51 / 4π * E_51

# ST phase
t_ST = [1e2, 1e3, 1e4] * t_ED
R_ST = ξ_ST * (1e51 * E_51 * (t_ST * Myr).^2 / μ / n_0).^0.2 / pc
v_ST = 0.4 * R_ST ./ t_ST * pc_Myr / km_s
M_ST = 4π/3 * μ * n_0 * (R_ST * pc).^3 / Msol

# Shell formation
t_sf = 4.4e-2 * E_51^0.22 * n_0^-0.55
v_sf = 202 * E_51^0.07 * n_0^0.13
R_sf = 22.6 * E_51^0.29 * n_0^-0.42
p_sf = 2.519e4 * E_51^0.93 * n_0^-0.13

# MCS phase
t_MCS = [1e1, 1e3, 1e5] * t_sf
R_MCS = ξ_MCS * (4π * p_sf * Msol * km_s * t_MCS * Myr / μ / n_0).^0.25 / pc
v_MCS = 0.25 * R_MCS ./ t_MCS * pc_Myr / km_s
M_MCS = 4π/3 * μ * n_0 * (R_MCS * pc).^3 / Msol

######## Superbubble
pdot_SB = α_p * M_ej * Msol * v_ED * km_s / (Δt_SB * Myr)
# Weaver Wind phase
t_W = [1e1, 1e2, 1e3] * Δt_SB
R_W = ξ_W * (1e51 * E_51 / (Δt_SB * Myr) * (t_W * Myr).^3 / μ / n_0).^0.2 / pc
v_W = 0.6 * R_W ./ t_W * pc_Myr / km_s
M_W = 4π/3 * μ * n_0 * (R_W * pc).^3 / Msol

# MCS phase
t_MDW = [1e2, 1e3, 1e4] * Δt_SB
R_MDW = ξ_MDW * (pdot_SB / μ / n_0)^0.25 * (t_MDW * Myr).^0.5 / pc
v_MDW = 0.5 * R_MDW ./ t_MDW * pc_Myr / km_s
M_MDW = 4π/3 * μ * n_0 * (R_MDW * pc).^3 / Msol

######## Starburst
pdot_StB = α_p * M_ej * Msol * v_ED * km_s / (Δt_StB * Myr)
# Ejecta Dominated Phase
t_ED_StB = 2.61e-6 * M_ej^(5/4) * E_51^(-3/4) / sqrt(n_0 * Δt_StB)

# Weaver Wind phase
t_W_StB = [1e2, 1e3, 1e4]
R_W_StB = ξ_W * (1e51 * E_51 / (Δt_StB * Myr) * (t_W_StB * Myr).^3 / μ / n_0).^0.2 / pc
v_W_StB = 0.6 * R_W_StB ./ t_W_StB * pc_Myr / km_s

# shell formation
t_sf_StB = 1.7e-2 * (E_51 / Δt_StB)^0.28 * n_0^-0.71
R_sf_StB = ξ_W * (1e51 * E_51 / (Δt_StB * Myr) * (t_sf_StB * Myr).^3 / μ / n_0).^0.2 / pc
M_sf_StB = μ * n_0 * (R_sf_StB * pc)^3 / 3 / Msol
p_sf_StB = 3/5 * M_sf_StB * R_sf_StB / t_sf_StB * pc_Myr / km_s + pdot_StB / 4π / Msol / km_s * Myr * t_sf_StB
p_sf_StB_SN = p_sf_StB / (1 + t_sf_StB / Δt_StB)

# MCS phase
t_MDW_StB = [1e2, 1e3, 1e4]
R_MDW_StB = ξ_MDW * (pdot_StB / μ / n_0)^0.25 * (t_MDW_StB * Myr).^0.5 / pc
v_MDW_StB = 0.5 * R_MDW_StB ./ t_MDW_StB * pc_Myr / km_s

# Examples

## Uniform Medium

### Definitions

In [ ]:
# get object storing results
Uniform_Medium = Dict{String, Dict{Symbol, Vector{Float64}}}()

In [ ]:
# define snapshot times
t_uniform = LogRange(log10(t_ED)-2, log10(14) + 3, 10000);

### Run Models

Below is an example of running the above defined model in the simplest possible environment: a uniform, stationary medium.   
In order to avoid lots of repetitive code, the actual model evaluation and subsequent post-processing to obtain easily plottable outputs is placed within a wrapper function:

```Julia
function run_model_uniform(model::Model, time::Union{AbstractRange{<:Real}, Vector{<:Real}}; N_SN=1.0)
    # run model
    t, x, v, n, M, f = numerical_solution(time, model=model, cosθ=0.0, ϕ=0.0) <----- This is the actual call doing the heavy lifting

    # Everything below is just post-processing of the outputs

    # number of SNe as a function of time
    Num_SN = input2func(N_SN)
    E_SN   = 1e51 / 4π * input2func(model.E_51)

    # store output
    output = Dict{Symbol, Vector{Float64}}()
    output[:r]     = [pos[1] for pos in x[:, 1]]
    output[:v]     = [pos[1] for pos in v[:, 1]]
    output[:dA]    = [pos[1] for pos in n[:, 1]]
    output[:t]     = t[:, 1]
    output[:M]     = M[:, 1]
    output[:V]     = [x[i, 1]'n[i, 1]/3 for i in eachindex(x[:, 1])]
    output[:p]     = @. output[:M] * output[:v] / Num_SN(t[:, 1])
    output[:f_kin] = @. 0.5 * output[:M] * output[:v]^2 * km_s^2 * Msol / E_SN.(t[:, 1])

    return output
end
```

Outputs are arrays, where the first dimension is time, and the second dimension corresponds to the set of directions for which the calculations are performed.   
(Exception: completion flag `f`, which only has the direction dimension).

In [ ]:
# run all models
# single SN
Uniform_Medium["SN_no_cool"] = run_model_uniform(SN_no_cool, t_uniform)
Uniform_Medium["SN"] = run_model_uniform(SN, t_uniform)

# Superbubble
Uniform_Medium["SB_no_cool"] = run_model_uniform(SB_no_cool, t_uniform, N_SN=N_SN_SB_continuous)
Uniform_Medium["SB_cont"] = run_model_uniform(SB_cont, t_uniform, N_SN=N_SN_SB_continuous)
Uniform_Medium["SB_bursty"] = run_model_uniform(SB_bursty, t_uniform, N_SN=N_SN_SB_bursty)

### Plot model results

In [ ]:
# some useful definitions
model_names = ["SN_no_cool", "SN", "SB_no_cool", "SB_cont"]
names       = Dict("SN_no_cool" => "single expl.", "SN" => "single expl. + cool.", "SB_no_cool" => "cont. inj.", "SB_cont" => "cont. inj. + cool.", "SB_bursty" => "discr. inj. + cool.")
c_model     = Dict("SN_no_cool" => "red", "SN" => "lightblue", "SB_no_cool" => "darkred", "SB_cont" => "darkblue", "SB_bursty" => "gold")

#### single explosion vs. continuous

The cell below generates a plot with five panels, showing the shock radius, expansion speed, accumulated mass, momentum per SN and the fraction of retained kinetic energy for the different models computed above.   
I will use these kinds of plots a lot throughout this document to highlight certain limits and behavior of the solutions.   
Here I am just demonstrating that the model reproduces a number of familiar regimes of blastwave evolution in uniform media.   

In [ ]:
# axis limits
tlim = [1e-2 * t_ED, 1.4e4]
rlim = [0.01 * R_ED, 3.3e4]
vlim = [0.01, 2 * v_ED]
mlim = [4, 5e9]
plim = [2π * p_ED, 1e3 * p_sf]
Elim = [1e-4, 1.5];

# create the figure and axes to plot the results
fig, axs = create_figure_uniform(Uniform_Medium, model_names, names, c_model, tlim)

# add y-axis limits
axs[1,1].set_ylim(rlim...)
axs[1,2].set_ylim(vlim...)
axs[1,3].set_ylim(mlim...)
axs[2,1].set_ylim(plim...)
axs[2,2].set_ylim(Elim...)

# Annotations
# Radius
## Timescales
add_timescale_annotation(axs[1,1], t_ED,  label=L"\mathrm{t_{ED \rightarrow ST}}", ylabel=1e4)
add_timescale_annotation(axs[1,1], t_sf,  label=L"\mathrm{t_{sf}}", ylabel=1e4)
add_timescale_annotation(axs[1,1], Δt_SB, label=L"\mathrm{\Delta t_{SN}}", ylabel=1e4)

## Asymptotic behavior
t = [1e-2, 1e-1, 1e0] * t_ED
add_line_annotation(axs[1,1], t, v_ED * t * km_s * Myr / pc, label=L"\mathrm{\propto t}")
add_line_annotation(axs[1,1], t_ST, R_ST, label=L"\mathrm{\propto t^{2/5}}")
add_line_annotation(axs[1,1], t_MCS, R_MCS, label=L"\mathrm{\propto t^{1/4}}")
add_line_annotation(axs[1,1], t_W, R_W, label=L"\mathrm{\propto t^{3/5}}")
add_line_annotation(axs[1,1], t_MDW, R_MDW, label=L"\mathrm{\propto t^{1/2}}")

# Speed
## Timescales
add_timescale_annotation(axs[1,2], t_ED,  label=L"\mathrm{t_{ED \rightarrow ST}}", ylabel=1e0)
add_timescale_annotation(axs[1,2], t_sf,  label=L"\mathrm{t_{sf}}", ylabel=1e0)
add_timescale_annotation(axs[1,2], Δt_SB, label=L"\mathrm{\Delta t_{SN}}", ylabel=1e0)
## Asymptotic behavior
add_horizontal_annotation(axs[1,2], v_ED, label=L"\mathrm{v_{ED}}", xlabel=0.1 * t_ED)
add_line_annotation(axs[1,2], t_ST, v_ST, label=L"\mathrm{\propto t^{-3/5}}")
add_line_annotation(axs[1,2], t_MCS, v_MCS, label=L"\mathrm{\propto t^{-3/4}}")
add_line_annotation(axs[1,2], t_W, v_W, label=L"\mathrm{\propto t^{-2/5}}")
add_line_annotation(axs[1,2], t_MDW, v_MDW, label=L"\mathrm{\propto t^{-1/2}}")

# Mass
## Timescales
add_timescale_annotation(axs[1,3], t_ED,  label=L"\mathrm{t_{ED \rightarrow ST}}", ylabel=1e7)
add_timescale_annotation(axs[1,3], t_sf,  label=L"\mathrm{t_{sf}}", ylabel=1e7)
add_timescale_annotation(axs[1,3], Δt_SB, label=L"\mathrm{\Delta t_{SN}}", ylabel=1e7)

## Asymptotic behavior
add_horizontal_annotation(axs[1,3], 2 * M_ej, label=L"\mathrm{2 \times M_{ej}}", xlabel=0.1 * t_ED)
add_line_annotation(axs[1,3], t_ST,  M_ST,  label=L"\mathrm{\propto t^{6/5}}")
add_line_annotation(axs[1,3], t_MCS, M_MCS, label=L"\mathrm{\propto t^{3/4}}")
add_line_annotation(axs[1,3], t_W,   M_W,   label=L"\mathrm{\propto t^{9/5}}")
add_line_annotation(axs[1,3], t_MDW, M_MDW, label=L"\mathrm{\propto t^{3/2}}")

# Momentum
## Timescales
add_timescale_annotation(axs[2,1], t_ED,  label=L"\mathrm{t_{ED \rightarrow ST}}", ylabel=1e7)
add_timescale_annotation(axs[2,1], t_sf,  label=L"\mathrm{t_{sf}}", ylabel=1e7)
add_timescale_annotation(axs[2,1], Δt_SB, label=L"\mathrm{\Delta t_{SN}}", ylabel=1e7)

## Asymptotic behavior
add_horizontal_annotation(axs[2,1],       4π * p_ED, label=L"\mathrm{p_{SN}}", xlabel=0.1 * t_ED)
add_horizontal_annotation(axs[2,1],       4π * p_sf, label=L"\mathrm{p_{sf}}", xlabel=0.1 * t_ED)
add_horizontal_annotation(axs[2,1], α_p * 4π * p_ED, label=L"\mathrm{\alpha_{p} \times p_{SN}}", xlabel=0.1 * t_ED)
add_line_annotation(axs[2,1], t_ST, M_ST .* v_ST,  label=L"\mathrm{\propto t^{3/5}}")
add_line_annotation(axs[2,1], t_W,  M_W .* v_W ./ N_SN_SB_continuous.(t_W),   label=L"\mathrm{\propto t^{2/5}}")

# Energy
## Timescales
add_timescale_annotation(axs[2,2], t_ED,  label=L"\mathrm{t_{ED \rightarrow ST}}", ylabel=1e-3)
add_timescale_annotation(axs[2,2], t_sf,  label=L"\mathrm{t_{sf}}", ylabel=1e-3)
add_timescale_annotation(axs[2,2], Δt_SB, label=L"\mathrm{\Delta t_{SN}}", ylabel=1e-3)

## Asymptotic behavior
add_horizontal_annotation(axs[2,2], 32/33, label=L"\mathrm{32/33}", xlabel=0.1 * t_ED)
add_horizontal_annotation(axs[2,2], 16/21, label=L"\mathrm{16/21}", xlabel=1e3)
add_horizontal_annotation(axs[2,2], 32/77, label=L"\mathrm{32/77}", xlabel=1e3)
add_line_annotation(axs[2,2], t_MCS, 0.5 * (4π * p_sf * Msol * km_s)^2 ./ (4π * E_ED * M_MCS * Msol),  label=L"\mathrm{\propto t^{-3/4}}")
add_line_annotation(axs[2,2], t_MDW,  0.5 * (α_p * 4π * p_ED * Msol * km_s .* t_MDW).^2 ./ (1e51 * SB_cont.E_51.(t_MDW) .* M_MDW * Msol),   label=L"\mathrm{\propto t^{-1/2}}")

#### discrete vs. continuous injection

Another one of these "global plots", demonstrating that continuous and discrete models have very similar behavior once enough SNe have exploded.

In [ ]:
# axis limits
tlim = [1e-2 * t_ED, 1.4e4]
rlim = [0.01 * R_ED, 5e3]
vlim = [0.1, 2 * v_ED]
mlim = [4, 5e9]
plim = [2π * p_ED, 1.5 * 4π * p_sf]
Elim = [1e-4, 1.5];

# create the figure and axes to plot the results
fig, axs = create_figure_uniform(Uniform_Medium, ["SB_cont", "SB_bursty"], names, c_model, tlim)

# add y-axis limits
axs[1,1].set_ylim(rlim...)
axs[1,2].set_ylim(vlim...)
axs[1,3].set_ylim(mlim...)
axs[2,1].set_ylim(plim...)
axs[2,2].set_ylim(Elim...)

# Annotations
# Radius
## Timescales
add_timescale_annotation(axs[1,1], t_ED,  label=L"\mathrm{t_{ED \rightarrow ST}}", ylabel=1e4)
add_timescale_annotation(axs[1,1], t_sf,  label=L"\mathrm{t_{sf}}", ylabel=1e4)
add_timescale_annotation(axs[1,1], Δt_SB, label=L"\mathrm{\Delta t_{SN}}", ylabel=1e4)

# Speed
## Timescales
add_timescale_annotation(axs[1,2], t_ED,  label=L"\mathrm{t_{ED \rightarrow ST}}", ylabel=1e0)
add_timescale_annotation(axs[1,2], t_sf,  label=L"\mathrm{t_{sf}}", ylabel=1e0)
add_timescale_annotation(axs[1,2], Δt_SB, label=L"\mathrm{\Delta t_{SN}}", ylabel=1e0)

# Mass
## Timescales
add_timescale_annotation(axs[1,3], t_ED,  label=L"\mathrm{t_{ED \rightarrow ST}}", ylabel=1e7)
add_timescale_annotation(axs[1,3], t_sf,  label=L"\mathrm{t_{sf}}", ylabel=1e7)
add_timescale_annotation(axs[1,3], Δt_SB, label=L"\mathrm{\Delta t_{SN}}", ylabel=1e7)

# Momentum
## Timescales
add_timescale_annotation(axs[2,1], t_ED,  label=L"\mathrm{t_{ED \rightarrow ST}}", ylabel=1e7)
add_timescale_annotation(axs[2,1], t_sf,  label=L"\mathrm{t_{sf}}", ylabel=1e7)
add_timescale_annotation(axs[2,1], Δt_SB, label=L"\mathrm{\Delta t_{SN}}", ylabel=1e7)

# Energy
## Timescales
add_timescale_annotation(axs[2,2], t_ED,  label=L"\mathrm{t_{ED \rightarrow ST}}", ylabel=1e-3)
add_timescale_annotation(axs[2,2], t_sf,  label=L"\mathrm{t_{sf}}", ylabel=1e-3)
add_timescale_annotation(axs[2,2], Δt_SB, label=L"\mathrm{\Delta t_{SN}}", ylabel=1e-3)

## Vertical Stratification

This is the first "non-trivial" example showcased in this document.   
I start by defining the environment and then run the model with a wrapper-function very similar to the one above.  

### Definitions

These scales are used to define the environment.

In [ ]:
# scale height
σ_ISM = 10.0                        # Velocity dispersion in km/s
z_s   = 33.73 * σ_ISM / sqrt(n_0)   # Scale height in pc

Here I define the Environment by overwriting only those fields, that matter for the vertically stratified medium: The density field and the gravitational field.   
   
I also define a condition for stopping the calculation early: once the flow reaches the midplane, i.e. once the blastwave has fully collapsed. 

In [ ]:
env_strat = Environment(density=(x::Vector{<:Real}, t::Real) -> cosh(x[3] / z_s)^-2, 
                        g_ext=(x::Vector{<:Real}, t::Real) -> -2 * tanh(x[3] / z_s) * σ_ISM^2 / z_s * [0, 0, 1],
                        out_of_bounds=(u::Vector{<:Real}, t::Real) -> u[3] < 0.0) # stop when flow reaches midplane

The gravitational potential is not used by the calculation, but it can be used to analyze the energetics:

In [ ]:
# define gravitational potential for potential energy calculation
logcosh(x::Real) = x < 100 ? log(cosh(x)) : x - log(2)
Φ_strat(z::Real)   = 2 * logcosh(z / z_s) * σ_ISM^2 * km_s^2

These scales are just used for plotting.

In [ ]:
# Charcteristic scales
# Free-fall timescale in Myr
t_ff  = 44.9 * n_0^-0.5

# Asymptotic mass
M_∞ = π^2 / 12 * μ * n_0 * (z_s * pc)^3 / Msol

# characteristic scales for single explosion (assuming MCS)
t_s = 1.4e3 * (σ_ISM / 10.0)^4 * n_0^-0.87 * E_51^-0.93  # Time it'd take to reach the scale height
v_s = 0.25 * z_s / t_s * pc_Myr / km_s                   # velocity at breakout
v_∞ = p_sf / M_∞                                         # asymptotic velocity after breakout

# characteristic scales for superbubble (assuming cooling)
t_s_SB = 6.64 * t_ff * (σ_ISM / 10)^2 * (α_p * sqrt(E_51 * M_ej)/ Δt_SB)^(-0.5)
v_s_SB = 0.5 * z_s / t_s_SB * pc_Myr / km_s
t_∞_SB       = [5e0, 1e1, 1e2] * t_s_SB
v_∞_SB       = α_p * p_ED / M_∞ * t_∞_SB / Δt_SB

# characteristic scales for starburst
t_s_StB  = 0.388 * t_ff * (σ_ISM / 10.0)^(5/3) * (E_51 / Δt_StB)^(-1/3)
v_s_StB  = 0.6 * z_s / t_s_StB * pc_Myr / km_s
t_s_MDW_StB = 6.64 * t_ff * (σ_ISM / 10.0)^2 * (α_p * sqrt(E_51 * M_ej)/ Δt_StB)^(-0.5)
v_s_MDW_StB = 0.5 * z_s / t_s_MDW_StB * pc_Myr / km_s
t_∞_MDW_StB = [5e0, 1e1, 1e2] * t_s_MDW_StB
v_∞_MDW_StB = α_p * p_ED / M_∞ * t_∞_MDW_StB / Δt_StB
t_∞ = 4π * M_∞ / M_ej * Δt_StB          # Time at which Ejecta match M_∞

In [ ]:
# get object storing results
Stratified_Medium = Dict{String, Dict{Symbol, Vector{Float64}}}()
Uniform_Medium = Dict{String, Dict{Symbol, Vector{Float64}}}() # for comparison

In [ ]:
# define snapshot times
t_strat = LogRange(log10(t_ED)-2, log10(14) + 3, 10000);

### Run Models

As before for the uniform medium, I am using a wrapper function:


```Julia
function run_model_strat(model::Model, environment::Environment, time::Union{AbstractRange{<:Real}, Vector{<:Real}}; N_SN=1.0, potential::Function=z->0.0)
    # run model
    t, x, v, n, M, f = numerical_solution(time, model=model, environment=environment, cosθ=1.0, ϕ=0.0) <---- Note the different call signature here

    # Everything below is just post-processing of the outputs
    
    # number of SNe as a function of time
    Num_SN = input2func(N_SN)
    E_SN   = 1e51 / 4π * input2func(model.E_51)

    # store output
    output = Dict{Symbol, Vector{Float64}}()
    output[:r]     = [pos[3] for pos in x[:, 1]] <-------- Since the medium is stratified in z-direction we extract the 3rd vector component here 
    output[:v]     = [pos[3] for pos in v[:, 1]]
    output[:dA]    = [pos[3] for pos in n[:, 1]]

    ...

    return output
end
```

For reference I am also running a uniform medium (same as above).

In [ ]:
# run all reference models
Uniform_Medium["SN"]  = run_model_uniform(SN, t_strat)
Uniform_Medium["SB"]  = run_model_uniform(SB_cont, t_strat, N_SN=N_SN_SB_continuous)
Uniform_Medium["StB"] = run_model_uniform(StB_cont, t_strat, N_SN=N_SN_StB_continuous)

In [ ]:
# run all models
Stratified_Medium["SN"]  = run_model_strat(SN, env_strat, t_strat, potential=Φ_strat)
Stratified_Medium["SB"]  = run_model_strat(SB_cont, env_strat, t_strat, N_SN=N_SN_SB_continuous, potential=Φ_strat)
Stratified_Medium["StB"] = run_model_strat(StB_cont, env_strat, t_strat, N_SN=N_SN_StB_continuous, potential=Φ_strat)

### Plot model results

In [ ]:
# some useful definitions
model_names = ["SN", "SB", "StB"]
names       = Dict("SN" => "Single SN", "SB" => L"\mathrm{Superbubble \ (\Delta t_{SN} = 1 \ Myr)}", "StB" => L"\mathrm{Starburst \ (\Delta t_{SN} = 100 \ yr)}")
c_model     = Dict("SN" => "gray", "SB" => "royalblue", "StB" => "darkred")

# axis limits
tlim = [1e-2 * t_ED_StB, 1.4e4]
rlim = [0.01 * R_ED, 1e4 * z_s]
vlim = [0.5, 2 * α_p * v_ED]
mlim = [0.3, 4e7]
plim = [0.5 * p_ED, 3 * p_sf]
Elim = [1e-2, 20];

Below is again one of those "global plots", showcasing that the single SN and Superbubble models both collapse within a single free-fall timescale.   
Only the starburst breaks out of the disk and reaches an asymptotic wind-speed. 

In [ ]:
# create the figure and axes to plot the results
fig, axs = create_figure_strat(Stratified_Medium, model_names, names, c_model, tlim)
plot_reference_models!(axs, Uniform_Medium, model_names, c_model)

# add y-axis limits
axs[1,1].set_ylim(rlim...)
axs[1,2].set_ylim(vlim...)
axs[1,3].set_ylim(mlim...)
axs[2,1].set_ylim(plim...)
axs[2,2].set_ylim(Elim...)

# Annotations
# Radius
## Timescales
add_timescale_annotation(axs[1,1], t_s, color=c_model["SN"])
add_timescale_annotation(axs[1,1], t_s_SB, color=c_model["SB"])
add_timescale_annotation(axs[1,1], t_s_StB, color=c_model["StB"])
add_timescale_annotation(axs[1,1], t_ff)

## Asymptotic behavior
add_horizontal_annotation(axs[1,1], z_s, label=L"\mathrm{H_{s}}", xlabel=0.1 * t_ED)

# Speed
## Timescales
add_timescale_annotation(axs[1,2], t_s, label=L"\mathrm{t_{s, SN}}", ylabel=1e3, color=c_model["SN"])
add_timescale_annotation(axs[1,2], t_s_SB, label=L"\mathrm{t_{s, SB}}", ylabel=1e3, color=c_model["SB"])
add_timescale_annotation(axs[1,2], t_s_StB, label=L"\mathrm{t_{s, StB}}", ylabel=1e3, color=c_model["StB"])
add_timescale_annotation(axs[1,2], t_ff, label=L"\mathrm{t_{ff}}", ylabel=1e4)

## Asymptotic behavior
add_horizontal_annotation(axs[1,2], α_p * v_ED, label=L"\mathrm{α_{p} \times v_{ED}}", xlabel=0.25 * t_ED)

# Mass
## Timescales
add_timescale_annotation(axs[1,3], t_s, label=L"\mathrm{t_{s, SN}}", ylabel=1e2, color=c_model["SN"])
add_timescale_annotation(axs[1,3], t_s_SB, label=L"\mathrm{t_{s, SB}}", ylabel=1e2, color=c_model["SB"])
add_timescale_annotation(axs[1,3], t_s_StB, label=L"\mathrm{t_{s, StB}}", ylabel=1e2, color=c_model["StB"])
add_timescale_annotation(axs[1,3], t_ff, label=L"\mathrm{t_{ff}}", ylabel=1e3)

## Asymptotic behavior
add_horizontal_annotation(axs[1,3], M_∞, label=L"\mathrm{M_{\infty}}", xlabel=0.1 * t_ED)

# Momentum
## Timescales
add_timescale_annotation(axs[2,1], t_s, label=L"\mathrm{t_{s, SN}}", ylabel=2 * p_sf, color=c_model["SN"])
add_timescale_annotation(axs[2,1], t_s_SB, label=L"\mathrm{t_{s, SB}}", ylabel=2 * p_sf, color=c_model["SB"])
add_timescale_annotation(axs[2,1], t_s_StB, label=L"\mathrm{t_{s, StB}}", ylabel=2 * p_sf, color=c_model["StB"])
add_timescale_annotation(axs[2,1], t_ff, label=L"\mathrm{t_{ff}}", ylabel=2/3 * p_sf)

## Asymptotic behavior
add_horizontal_annotation(axs[2,1],       p_ED, label=L"\mathrm{p_{SN}}", xlabel=0.1 * t_ED)
add_horizontal_annotation(axs[2,1],       p_sf, label=L"\mathrm{p_{sf}}", xlabel=0.1 * t_ED)
add_horizontal_annotation(axs[2,1], α_p * p_ED, label=L"\mathrm{\alpha_{p} \times p_{SN}}", xlabel=0.25 * t_ED)

# Energy
## Timescales
add_timescale_annotation(axs[2,2], t_s, color=c_model["SN"])
add_timescale_annotation(axs[2,2], t_s_SB, color=c_model["SB"])
add_timescale_annotation(axs[2,2], t_s_StB, color=c_model["StB"])
add_timescale_annotation(axs[2,2], t_ff)

## Differential Rotation / Shear

Next up is the second "non-trivial" example: A blastwave in a differentially rotating medium.   
I start by defining the environment and then run the model with a wrapper-function very similar to the examples above.   
This is the first example, where I will show how to sample multiple directions at once (full sphere and slices).    

### Definitions

In [ ]:
# Environment parameters
V_rot = 200.0            # Disk rotation velocity in km/s
R_gal = 8000.0           # Galactocentric radius in pc
σ_ISM = 10.0             # Velocity dispersion in km/s

# Scales related to galactic rotation
t_orb = 2π * R_gal / V_rot * pc_Myr / km_s
Ω_Myr = 2π / t_orb
p_4 = p_ED * 4π / 1e4

v_shear_SN  = 1.58 * (t_orb / 100)^(-0.75) * E_51^0.23 * n_0^-0.28
v_shear_StB  = 2.29 * (t_orb / 100)^(-0.5) * (α_p * p_4 / Δt_SB)^0.25 * n_0^-0.25
v_shear_StB = 2.29 * (t_orb / 100)^(-0.5) * (α_p * p_4 / Δt_StB)^0.25 * n_0^-0.25

# epicycle timescale
t_κ  = t_orb / √2

Below I define the environment from scratch: I start from the default environment `Environment()` and then overwrite all the different fields.   
For differential rotation the relevant fields are the external velocity, the velocity gradient, the center of the bubble, and the corresponding gravitational field.    

In [ ]:
# define environment
env_shear = Environment()

# stratified atmosphere
env_shear.density = (x::Vector{<:Real}, t::Real) -> 1.0           # in units of scale-density (n_0)
env_shear.v_ext   = (x::Vector{<:Real}, t::Real) -> [V_rot / max(sqrt(x[1]^2 + x[2]^2), SMALL) * [-x[2], x[1]]..., 0.0] # in km/s

# velocity gradient
"The external velocity gradient function."
function ∇v_ext(x::Vector{<:Real}, t::Real)::Matrix{<:Real}
    invR = 1.0 / max(sqrt(x[1]^2 + x[2]^2), SMALL)

    M = [x[1] * x[2] -x[1]^2 0; 
         x[2]^2 -x[1] * x[2] 0; 
          0 0 0]

    M[1:2, 1:2] *= V_rot * invR^3

    return M
end

env_shear.dv_ext_dt     = (x::Vector{<:Real}, t::Real) -> zeros(3)
env_shear.x_c           = (t::Real) -> 8000.0 * [cos(Ω_Myr * t), sin(Ω_Myr * t), 0]
env_shear.∇v_ext        = ∇v_ext # in km/s / pc
env_shear.g_ext         = (x::Vector{<:Real}, t::Real) -> -V_rot^2 / max(x[1]^2 + x[2]^2, SMALL) * [x[1:2]..., 0.0]
env_shear.P_CR          = (x::Vector{<:Real}, t::Real) -> 0.5 * σ_ISM^2        # in units of scale-density × (km/s)^2 will be rescaled with f_CR of the model
env_shear.out_of_bounds = (u::Vector{<:Real}, t::Real) -> false          # No out-of-bounds stopping condition

For purposes of energy analysis only I also include the gravitational potential.

In [ ]:
# define gravitational potential for potential energy calculation
Φ_shear(R::Real) = log(R / R_gal) * V_rot^2 * km_s^2

In [ ]:
# get object storing results
Shearing_Medium = Dict{String, Dict{Symbol, Array{Float64}}}()
Uniform_Medium = Dict{String, Dict{Symbol, Vector{Float64}}}() # for comparison

In [ ]:
# define snapshot times
t_shear = LogRange(-5, log10(0.7 * t_orb), 10000);

In [ ]:
# Healpix NSIDE (computation time scales like Nside^2)
Nside = 4

### 3D Geometry

#### Run Models

As with the examples above, I am using a wrapper function:


```Julia
function run_model_shear(model::Model, environment::Environment, time::Union{AbstractRange{<:Real}, Vector{<:Real}}; N_SN=1.0, potential::Function=z->0.0, Nside::Integer=4)
    # run model: using HealpixMap to sample the entire sphere.
    t, x, v, n, M, f = numerical_solution(time, model=model, environment=environment, map=HealpixMap{Float64, NestedOrder}(Nside))

    # Everything below is just post-processing of the outputs
    dΩ = 4π / (Nside^2 * 12) # area associated with Healpix NSIDE
    ...
    
    # e.g., Here I am computing some summary statistics over the whole surface at each point in time
    for it in eachindex(time)
        ...
        Mass_shear[it]   = sum(M[it, :]) * dΩ <---- remember that M in the model is actually dM/dΩ
        ...
    end

    ...

    return output
end
```
For large Nside this can take a while.   
BUT: `numerical_solution` uses multi-threading under the hood to parallelize the computation for different directions, and thus should scale perfectly with more threads. 

In [ ]:
# run all reference models
Uniform_Medium["SN"]  = run_model_uniform(SN, t_shear)
Uniform_Medium["SB"]  = run_model_uniform(SB_cont, t_shear, N_SN=N_SN_SB_continuous)
Uniform_Medium["StB"] = run_model_uniform(StB_cont, t_shear, N_SN=N_SN_StB_continuous)

In [ ]:
# run all models
Shearing_Medium["SN"]  = run_model_shear(SN, env_shear, t_shear, potential=Φ_shear, Nside=Nside)
Shearing_Medium["SB"]  = run_model_shear(SB_cont, env_shear, t_shear, N_SN=N_SN_SB_continuous, potential=Φ_shear, Nside=Nside)
Shearing_Medium["StB"] = run_model_shear(StB_cont, env_shear, t_shear, N_SN=N_SN_StB_continuous, potential=Φ_shear, Nside=Nside)

#### Plot model results

In [ ]:
# some useful definitions
model_names = ["SN", "SB", "StB"]
names       = Dict("SN" => "Single SN", "SB" => L"\mathrm{Superbubble \ (\Delta t_{SN} = 1 \ Myr)}", "StB" => L"\mathrm{Starburst \ (\Delta t_{SN} = 100 \ yr)}")
c_model     = Dict("SN" => "gray", "SB" => "royalblue", "StB" => "darkred")

##### Globals Plot

Below is yet another one of those "global plots".   
This one showcases how blastwave expansion is affected by differential rotation.   
Differences become most prominent after half an epicycle when radial motion starts to be converted to tangential motion.   
This leads to overall **smaller** remnants.

In [ ]:
# axis limits
tlim = [0.2 * t_orb / (8π), t_shear[end]]
rlim = [70.0, 9.9e3]
vlim = [0.02, 90]
mlim = [5e4, 1.5e10]
plim = [0.15 * p_ED * 4π, 2.5 * p_sf * 4π]
Elim = [1.5e-4, 0.9];

# create the figure and axes to plot the results
fig, axs = create_figure_shear(t_shear, Shearing_Medium, model_names, names, c_model, tlim)
plot_reference_models!(axs, Uniform_Medium, model_names, c_model, solid_angle=4π)

# add y-axis limits
axs[1,1].set_ylim(rlim...)
axs[1,2].set_ylim(vlim...)
axs[1,3].set_ylim(mlim...)
axs[2,1].set_ylim(plim...)
axs[2,2].set_ylim(Elim...)

# Annotations
# Radius
## Timescales
add_timescale_annotation(axs[1,1], t_orb / (8π), label=L"\mathrm{t_{shear, SN}}", ylabel=5e2, color=c_model["SN"])
add_timescale_annotation(axs[1,1], t_orb / (4π), label=L"\mathrm{t_{shear, StB}}", ylabel=5e2, color=c_model["SB"])
add_timescale_annotation(axs[1,1], 0.5t_κ, label=L"\mathrm{t_{κ}/2}", ylabel=5e2)
add_timescale_annotation(axs[1,1], 0.25t_κ, label=L"\mathrm{t_{κ}/4}", ylabel=5e2)

# Speed
## Timescales
add_timescale_annotation(axs[1,2], t_orb / (8π), label=L"\mathrm{t_{shear, SN}}", ylabel=1e-1, color=c_model["SN"])
add_timescale_annotation(axs[1,2], t_orb / (4π), label=L"\mathrm{t_{shear, StB}}", ylabel=1e-1, color=c_model["SB"])
add_timescale_annotation(axs[1,2], 0.5t_κ, label=L"\mathrm{t_{κ}/2}", ylabel=1e-1)
add_timescale_annotation(axs[1,2], 0.25t_κ, label=L"\mathrm{t_{κ}/4}", ylabel=1e-1)

# Mass
## Timescales
add_timescale_annotation(axs[1,3], t_orb / (8π), label=L"\mathrm{t_{shear, SN}}", ylabel=1e7, color=c_model["SN"])
add_timescale_annotation(axs[1,3], t_orb / (4π), label=L"\mathrm{t_{shear, StB}}", ylabel=1e7, color=c_model["SB"])
add_timescale_annotation(axs[1,3], 0.5t_κ, label=L"\mathrm{t_{κ}/2}", ylabel=1e7)
add_timescale_annotation(axs[1,3], 0.25t_κ, label=L"\mathrm{t_{κ}/4}", ylabel=1e7)

# Momentum
## Timescales
add_timescale_annotation(axs[2,1], t_orb / (8π), color=c_model["SN"])
add_timescale_annotation(axs[2,1], t_orb / (4π), color=c_model["SB"])
add_timescale_annotation(axs[2,1], 0.5t_κ)
add_timescale_annotation(axs[2,1], 0.25t_κ)

## Asymptotic behavior
add_horizontal_annotation(axs[2,1],       4π * p_sf, label=L"\mathrm{p_{sf}}", xlabel=t_orb / (16π))
add_horizontal_annotation(axs[2,1], 4π * α_p * p_ED, label=L"\mathrm{\alpha_{p} \times p_{SN}}", xlabel=t_orb / (16π))

# Energy
## Timescales
add_timescale_annotation(axs[2,2], t_orb / (8π), color=c_model["SN"])
add_timescale_annotation(axs[2,2], t_orb / (4π), color=c_model["SB"])
add_timescale_annotation(axs[2,2], 0.5t_κ)
add_timescale_annotation(axs[2,2], 0.25t_κ)

##### Geometry tracks

Below is a plot showing the trajectories of the models in the two-dimensional geometry phase-space.   
Differential rotation gradually deforms the remnants, making them increasingly prolate on a timescale of ~10% of an orbit.

In [ ]:
# Create figure showing bubble shapes
fig, ax = plot_geometry_tracks_shear(t_shear, Shearing_Medium, model_names, names, c_model, 0.1:0.1:0.7, t_orb)

##### Pitch Angles

Below is a plot showing the time evolution of the pitch angle of these models (angle between the rotation vector and the major axis of the remnant).     
Pitch angles start out at roughly 45 degrees and gradually decrease, aligning the remnants more and more with the rotation direction.   
Initially the remnants are still spherical, so the pitch angles are not exactly meaningful.   
At late stages the remnants are no longer convex, rendering the method of fitting an ellipsoid to measure the geometry increasingly unreliable.   
(These regimes are grayed out)

In [ ]:
# Create figure showing alignment
fig, ax = plot_pitch_angles_shear(t_shear, Shearing_Medium, model_names, names, c_model, 0.1:0.1:0.7, t_orb)

### Slices

#### Run models

Sometimes it can be useful to just look at slices (as in this case a slice through the xy-plane).   
This is also supported and again I am using a wrapper function for brevity:


```Julia
function run_model_shear_slice(model::Model, environment::Environment, time::Union{AbstractRange{<:Real}, Vector{<:Real}}; N_SN=1.0,
    φ_range::Union{AbstractRange{<:Real}, Vector{<:Real}}=LinRange(0, 2π, 5))
    # run model
    t, x, v, n, M, f = numerical_solution(time, model=model, environment=environment, cosθ=0.0, ϕ=φ_range) <---- cosθ & ϕ can be a single number or many numbers

    # number of SNe as a function of time
    Num_SN = input2func(N_SN)
    E_SN   = 1e51 / 4π * input2func(model.E_51)

    output = Dict{Symbol, Any}()
    output[:t] = t
    output[:x] = x
    output[:n] = n
    output[:v] = v
    output[:p] = M .* v ./ Num_SN.(t)

    return output
end
```
This (1D) runs much faster than sampling the whole sphere (2D) so one can even go to relatively high resolution without having to wait too long for the results.

In [ ]:
# get object storing results
Shearing_Medium_slices = Dict{String, Dict{Symbol, Any}}()
Uniform_Medium = Dict{String, Dict{Symbol, Vector{Float64}}}() # for comparison

# get azimuthal sampling
Nφ = 65
φ_range = LinRange(0, 2π, Nφ)

# evolve for slightly longer time
t_slice = LogRange(-5, log10(2 * t_orb), 10000);

In [ ]:
# run all reference models
Uniform_Medium["SN"]  = run_model_uniform(SN, t_slice)
Uniform_Medium["SB"]  = run_model_uniform(SB_cont, t_slice, N_SN=N_SN_SB_continuous)
Uniform_Medium["StB"] = run_model_uniform(StB_cont, t_slice, N_SN=N_SN_StB_continuous)

In [ ]:
# run all models
Shearing_Medium_slices["SN"]  = run_model_shear_slice(SN, env_shear, t_slice, φ_range=φ_range)
Shearing_Medium_slices["SB"]  = run_model_shear_slice(SB_cont, env_shear, t_slice, N_SN=N_SN_SB_continuous, φ_range=φ_range)
Shearing_Medium_slices["StB"] = run_model_shear_slice(StB_cont, env_shear, t_slice, N_SN=N_SN_StB_continuous, φ_range=φ_range)

#### Plot Model Results

In [ ]:
# some useful definitions
model_names = ["SN", "SB", "StB"]
names       = Dict("SN" => "Single SN", "SB" => L"\mathrm{Superbubble \ (\Delta t_{SN} = 1 \ Myr)}", "StB" => L"\mathrm{Starburst \ (\Delta t_{SN} = 100 \ yr)}")
c_model     = Dict("SN" => "gray", "SB" => "royalblue", "StB" => "darkred")

##### Contour Plot

The plot below visualizes the slices through the xy plane at different points in time to showcase how the remnants grow and at the same time are sheared apart.   
The residual velocity vectors are also shown, showcasing epicyclic oscillations.   

In [ ]:
# Create figure showing contours of shock surface in a slice through the midplane
times_frame = [0.001, 0.25, 0.5]
fig, axs = plot_contours_shear(t_slice, Shearing_Medium_slices, env_shear, model_names, names, c_model, times_frame, t_orb,
                               L_frame=[[125, 1125, 2000], [625, 1e4, 1.75e4]],  # Width of panels in pc [[top row (SN, SB)], [bottom row (StB)]] 
                               vel_scale=[[500, 15, 15], [4000, 200, 150]],      # Scale of velocity arrows in pc / Myr [[top row (SN, SB)], [bottom row (StB)]] 
                               N_arrow_sampling=32)                              # Number of arrows to plot

##### Momentum Decomposition

Below the blastwave momentum is shown for two directions (fully radial and fully azimuthal) to showcase how the momenta are affected by epicycles.   
In particular the momentum (here in units of the momentum of an equivalent blastwave of the same age in a uniform medium) gets coupled less efficiently to the blastwave for continuously driven blastwaves.

In [ ]:
# make momentum decomposition plots
fig, axs = plot_momentum_decomposition(t_slice, Shearing_Medium_slices, Uniform_Medium, env_shear, model_names, names, c_model, 2.0, t_orb, φ_range)

## Structured Medium - Filaments

The final "non-trivial" example is a blastwave in a structured medium.   
Here I want to showcase how to build complex environments from templates (In this case filaments) and how to study the effect of such environments.  

### Definitions

### Blastwave approaching single Filament

#### Definitions

Here an environment with a single Filament is defined.  
As for the shearing medium, I start from the default environment and overwrite all fields appropriately.   

In [ ]:
# Environment parameters
T_fil_single = 600                                 # Temperature of filament in K
σ_fil_single = sqrt(kB * T_fil_single / μ) / km_s  # Velocity dispersion of filament in K
n_fil_single = 10                                  # Central density of filament in amu/cc
σ_ISM = 10.0                                       # Velocity dispersion of background medium in km/s

# characteristic scales
δ_fil_single = n_fil_single / n_0                                   # Overdensity of filament
R_fil_single = 169.0 * 4 * (σ_fil_single / 10) / sqrt(n_fil_single) # Filament scale radius in pc
x_fil_single = [8 * R_fil_single, 0, 0]                               # Initial coordinates of explosion center, relative to filament

# Mass
M_fil = n_fil_single * x_fil_single[1]^2 * R_fil_single * μ * pc^3 / Msol # Maximum mass to be swept by blastwave approaching filament (in Msol / sr)
M_0   = 1/3 * n_0 * x_fil_single[1]^3 * μ * pc^3 / Msol                   # Maximum mass to be swept up in uniform medium with radius = "x_fil_single" (in Msol / sr)

In [ ]:
# define environment
env_fil_single = Environment()

# stratified atmosphere
env_fil_single.density = (x::Vector{<:Real}, t::Real) -> δ_fil_single * ρ_filament(x, R_fil=R_fil_single, x_fil=x_fil_single, dir_fil=[0, 1, 0]) + 1.0 # in units of scale-density (n_0)
env_fil_single.v_ext   = (x::Vector{<:Real}, t::Real) -> zeros(3) # in km/s
env_fil_single.dv_ext_dt     = (x::Vector{<:Real}, t::Real) -> zeros(3)
env_fil_single.x_c           = (t::Real) -> zeros(3)
env_fil_single.∇v_ext        = (x::Vector{<:Real}, t::Real) ->zeros(3, 3) # in km/s / pc
env_fil_single.g_ext         = (x::Vector{<:Real}, t::Real) -> σ_fil_single^2 / R_fil_single * g_filament(x, R_fil=R_fil_single, x_fil=x_fil_single, dir_fil=[0, 1, 0])
env_fil_single.P_CR          = (x::Vector{<:Real}, t::Real) -> 0.5 * σ_ISM^2        # in units of scale-density × (km/s)^2 will be rescaled with f_CR of the model
env_fil_single.out_of_bounds = (u::Vector{<:Real}, t::Real) -> u[3] < -1e-3 || (abs(u[1]) >= x_fil_single[1]) # stop if the filament is reached or if z < 0
Φ_fil_single(x::Vector{<:Real})  = Φ_filament(x, R_fil=R_fil_single, x_fil=x_fil_single, dir_fil=[0, 1, 0]) * σ_fil_single^2 * km_s^2

In [ ]:
# get object storing results
Single_Filament = Dict{String, Dict{Symbol, Vector{Float64}}}()
Uniform_Medium  = Dict{String, Dict{Symbol, Vector{Float64}}}() # for comparison

# evolve for up to 100 Myr
t_fil = LogRange(-5, 2, 10000);

#### Run models

Here a wrapper function for a single direction is used:   

```Julia
function run_model_single_fil(model::Model, environment::Environment, time::Union{AbstractRange{<:Real}, Vector{<:Real}}; N_SN=1.0, potential::Function=x->0.0)
    # run model
    t, x, v, n, M, f = numerical_solution(time, model=model, environment=environment, cosθ=0.0, ϕ=0.0)

    # number of SNe as a function of time
    Num_SN = input2func(N_SN)
    E_SN   = 1e51 / 4π * input2func(model.E_51)

    # store output
    output = Dict{Symbol, Vector{Float64}}()
    output[:t]     = t[:, 1]
    output[:r]     = [pos[1] for pos in x[:, 1]]
    output[:v]     = [pos[1] for pos in v[:, 1]]
    output[:dA]    = [pos[1] for pos in n[:, 1]]
    output[:M]     = M[:, 1]
    output[:V]     = [x[i, 1]'n[i, 1]/3 for i in eachindex(x[:, 1])]
    output[:n_H]   = output[:M] * Msol ./ (output[:V] * pc^3) / μ 
    output[:p]     = @. output[:M] * output[:v] / Num_SN(t[:, 1])
    output[:f_kin] = @. 0.5 * output[:M] * output[:v]^2 * km_s^2 * Msol / E_SN.(t[:, 1])
    output[:f_pot] = output[:M] * Msol .* potential.(x[:, 1]) ./ E_SN.(t[:, 1])

    return output
end
```

In [ ]:
# run all reference models
Uniform_Medium["SN"]  = run_model_uniform(SN, t_fil)
Uniform_Medium["SB"]  = run_model_uniform(SB_cont, t_fil, N_SN=N_SN_SB_continuous)
Uniform_Medium["StB"] = run_model_uniform(StB_cont, t_fil, N_SN=N_SN_StB_continuous)

In [ ]:
# run all models
Single_Filament["SN"]  = run_model_single_fil(SN, env_fil_single, t_fil, potential=Φ_fil_single)
Single_Filament["SB"]  = run_model_single_fil(SB_cont, env_fil_single, t_fil, N_SN=N_SN_SB_continuous, potential=Φ_fil_single)
Single_Filament["StB"] = run_model_single_fil(StB_cont, env_fil_single, t_fil, N_SN=N_SN_StB_continuous, potential=Φ_fil_single)

#### Plot model results

Another one of these "global plots".   
This one showcases how a blastwave may be affected when it approaches a dense ISM structure (like a filament).   
Depending on the speed with which it encounters the structure the structure may be either overrun (starburst) or the shock begins to free-fall onto the structure, approaching the free-fall velocity.   
During this free-fall the density of the shock surface increases approaching the density of the overdensity, and may even overshoot due to converging streamlines.   

In [ ]:
# some useful definitions
model_names = ["SN", "SB", "StB"]
names       = Dict("SN" => "Single SN", "SB" => L"\mathrm{Superbubble \ (\Delta t_{SN} = 1 \ Myr)}", "StB" => L"\mathrm{Starburst \ (\Delta t_{SN} = 100 \ yr)}")
c_model     = Dict("SN" => "gray", "SB" => "royalblue", "StB" => "darkred")

# axis limits
tlim = [1e-2, t_fil[end]]
rlim = [11.0, 1.5 * x_fil_single[1]]
vlim = [1.5, 2e3]
mlim = [1.5e1, 1.5 * M_fil]
plim = [2 * p_ED, 3 * p_sf]
Elim = [1.5e-4, 0.9];
nlim = [0.9 * n_0, 2 * n_fil_single]

In [ ]:
# create the figure and axes to plot the results
fig, axs = create_figure_single_fil(Single_Filament, model_names, names, c_model, tlim)
plot_reference_models!(axs, Uniform_Medium, model_names, c_model, update_legend=false)

# add y-axis limits
axs[1,1].set_ylim(rlim...)
axs[1,2].set_ylim(vlim...)
axs[1,3].set_ylim(mlim...)
axs[2,1].set_ylim(plim...)
axs[2,2].set_ylim(Elim...)
axs[2,3].set_ylim(nlim...)

# Annotations
# Radius
add_horizontal_annotation(axs[1,1], x_fil_single[1], label=L"\mathrm{d}", xlabel=5e-2)

# Speed
add_horizontal_annotation(axs[1,2], 2 * σ_fil_single, label=L"\mathrm{2 \times σ_{fil}}", xlabel=5e-2)

# Mass
add_horizontal_annotation(axs[1,3], M_fil, label=L"\mathrm{M_{fil}}", xlabel=5e-2)
add_horizontal_annotation(axs[1,3], M_0, label=L"\mathrm{M_{0}}", xlabel=5e-2)

# Momentum
add_horizontal_annotation(axs[2,1],       p_sf, label=L"\mathrm{p_{sf}}", xlabel=5e-2)
add_horizontal_annotation(axs[2,1], α_p * p_ED, label=L"\mathrm{\alpha_{p} \times p_{SN}}", xlabel=5e-2)

# Density
add_horizontal_annotation(axs[2,3], n_0, label=L"\mathrm{n_{0}}", xlabel=5e-2)
add_horizontal_annotation(axs[2,3], n_fil_single, label=L"\mathrm{n_{fil}}", xlabel=5e-2)

### Blastwave between two Filaments

Here an environment with two filaments is defined, by simply calling two filament templates at different locations.   
As for the previous, I start from the default environment and overwrite all fields appropriately.   

#### Definitions

In [ ]:
# background medium
n_0_background = 0.1 / n_0            # Density of background medium
σ_ISM = 10.0                          # Velocity dispersion of background medium in km/s (only relevant for CRs)
x_fil          = [100, 0, 0]          # Initial coordinates of explosion center, relative to filaments

# Environment parameters
n_fil = 10 / n_0                                                                    # Central density of filament in code units
T_fil = 100 * (x_fil[1] / 53.62)^2 * n_fil / (sqrt(2 * n_fil / n_0_background) - 1) # Temperature of filament in K (set such that ρ(x_fil) ∼ ρ_background)
σ_fil = sqrt(kB * T_fil / μ) / km_s                                                 # Velocity dispersion of filament in km/s
R_fil  = 53.62 * sqrt(T_fil / 100) / sqrt(n_fil)                                    # Filament scale radius

In [ ]:
# define environment
env_fil = Environment()

# stratified atmosphere
env_fil.density = (x::Vector{<:Real}, t::Real) -> max(n_fil * (ρ_filament(x, R_fil=R_fil, x_fil=x_fil, dir_fil=[0, 1, 0]) + ρ_filament(x, R_fil=R_fil, x_fil=-x_fil, dir_fil=[0, 1, 0])), n_0_background)
env_fil.v_ext   = (x::Vector{<:Real}, t::Real) -> zeros(3) # in km/s
env_fil.dv_ext_dt     = (x::Vector{<:Real}, t::Real) -> zeros(3)
env_fil.x_c           = (t::Real) -> zeros(3)
env_fil.∇v_ext        = (x::Vector{<:Real}, t::Real) ->zeros(3, 3) # in km/s / pc
env_fil.g_ext         = (x::Vector{<:Real}, t::Real) -> σ_fil^2 / R_fil * (g_filament(x, R_fil=R_fil, x_fil=x_fil, dir_fil=[0, 1, 0]) + g_filament(x, R_fil=R_fil, x_fil=-x_fil, dir_fil=[0, 1, 0]))
env_fil.P_CR          = (x::Vector{<:Real}, t::Real) -> 0.5 * σ_ISM^2
env_fil.out_of_bounds = (u::Vector{<:Real}, t::Real) -> dot(cross(u[10:12], u[13:15]), u[1:3]) < 0
Φ_fil(x::Vector{<:Real})  = (Φ_filament(x, R_fil=R_fil, x_fil=x_fil, dir_fil=[0, 1, 0]) + Φ_filament(x, R_fil=R_fil, x_fil=-x_fil, dir_fil=[0, 1, 0])) * σ_fil^2 * km_s^2

#### Contours

##### Definitions

In [ ]:
# get object storing results
filament_slices = Dict{String, Dict{Symbol, Any}}()

# evolve for up to 100 Myr
t_fil = LogRange(-5, 2, 10000);

In [ ]:
Nφ         = 65
φ_range    = LinRange(0, 2π, Nφ)
cosθ_range = cos.(LinRange(0, π/2, Nφ));

##### Run models

Let's first build some intuition by looking at slices.   
Here a wrapper function for arbitrary subsamples of the sphere (e.g., slices):   

```Julia
function run_model_filaments_slice(model::Model, environment::Environment, time::Union{AbstractRange{<:Real}, Vector{<:Real}}; N_SN=1.0,
    cosθ::Union{Real, AbstractRange{<:Real}, Vector{<:Real}}=LinRange(-1, 1, 5), ϕ::Union{Real, AbstractRange{<:Real}, Vector{<:Real}}=LinRange(0, 2π, 5))
    # run model
    t, x, v, n, M, f = numerical_solution(time, model=model, environment=environment, cosθ=cosθ, ϕ=ϕ)

    # number of SNe as a function of time
    Num_SN = input2func(N_SN)
    E_SN   = 1e51 / 4π * input2func(model.E_51)

    output = Dict{Symbol, Any}()
    output[:t] = t
    output[:x] = x
    output[:n] = n
    output[:v] = v
    output[:p] = M .* v ./ Num_SN.(t)
    output[:case] = f

    return output
end
```

In [ ]:
# run all models (xy-plane)
filament_slices["SN_xy"] = run_model_filaments_slice(SN, env_fil, t_fil, cosθ=0.0, ϕ=φ_range)
filament_slices["SB_xy"] = run_model_filaments_slice(SB_cont, env_fil, t_fil, N_SN=N_SN_SB_continuous, cosθ=0.0, ϕ=φ_range)

In [ ]:
# run all models (xz-plane)
filament_slices["SN_xz"] = run_model_filaments_slice(SN, env_fil, t_fil, cosθ=cosθ_range, ϕ=0.0)
filament_slices["SB_xz"] = run_model_filaments_slice(SB_cont, env_fil, t_fil, N_SN=N_SN_SB_continuous, cosθ=cosθ_range, ϕ=0.0)

##### Plot Model Results

This plot illustrates the geometry of the remnant using slices through the xy & xz plane.   
The remnant can expand rapidly in the corridor between the filaments, but is stalled quickly in the directions of the filaments.   
Even vertically, the gravitational pull of the filaments halts the remnants to some degree, leading to a complex geometry.   

In [ ]:
# some useful definitions
model_names = ["SN", "SB"]
names = Dict("SN" => "Single SN", "SB" => L"\mathrm{Superbubble \ (\Delta t_{SN} = 1 \ Myr)}")
c_model = Dict("SN" => "gray", "SB" => "royalblue")

In [ ]:
# Create figure showing contours of shock surface in a slice through the midplane
times_frame = [1, 5, 10]  # Myr
fig, axs = plot_contours_filaments(t_fil, filament_slices, model_names, names, c_model, times_frame, x_fil,
                               L_frame=[600, 600, 600],  # Width of panels in pc (bottom row (xz) only half)
                               vel_scale=[250, 100, 75], # Scale of velocity arrows in pc / Myr (per panel)
                               N_arrow_sampling=32)      # Number of arrows to plot

#### Full Geometry

##### Definitions

In [ ]:
# get object storing results
Structured_Medium = Dict{String, Dict{Symbol, Array{Float64}}}()
Uniform_Medium = Dict{String, Dict{Symbol, Vector{Float64}}}() # for comparison

In [ ]:
# Healpix NSIDE (computation time scales like Nside^2)
Nside = 4

##### Run models

To analyze the complex geometry of the remnant it we need to sample to whole sphere.  
Here a wrapper function for another Healpix-sampled solution:   

```Julia
function run_model_filaments(model::Model, environment::Environment, time::Union{AbstractRange{<:Real}, Vector{<:Real}}; N_SN=1.0, potential::Function=x->0.0, Nside::Integer=4)
    # run model
    t, x, v, n, M, f = numerical_solution(time, model=model, environment=environment, map=HealpixMap{Float64, NestedOrder}(Nside))

    ...

    return output
end
```

In [ ]:
# run all models
Structured_Medium["SN"] = run_model_filaments(SN, env_fil, t_fil, potential=Φ_fil, Nside=Nside)
Structured_Medium["SB"] = run_model_filaments(SB_cont, env_fil, t_fil, N_SN=N_SN_SB_continuous, potential=Φ_fil, Nside=Nside)

An observer might capture the remnant at some time $t$ and measure the size, density, etc.   
However crucially, they would not have access to the density at times $< t$.   
So they would try to fit an analytical model for blastwaves assuming a constant ambient density corresponding to the density they measure.      
Let's do this exercise and appreciate how at known age $t$ this 'observational' reproduction might differ from the results above.   
For a radiative blastwave the equation of motion can be integrated to yield:

$$
 \frac{4\pi}{3} R^3 \frac{d R}{dt} = p_{sf} + \dot{p}_{in} t
$$

Where $\dot{p}_{in}=0$ for a single SN. 

In [ ]:
# Single SN
n_uniform = Structured_Medium["SN"][:n_H] # ambient density measured at each time

# characteristic scales (corresponding to these densities)
t_sf_SN = 4.4e-2  * n_uniform.^-0.55
R_sf_SN = 22.6    * n_uniform.^-0.42
p_sf_SN = 3.165e5 * n_uniform.^-0.13

# store output
output = Dict{Symbol, Vector{Float64}}()
output[:t]     = t_fil
output[:r]     = @. (R_sf_SN^4 + 3 / π * p_sf_SN * Msol * km_s / μ / n_uniform * max(t_fil - t_sf_SN, 0) * Myr / pc^4)^0.25 # Analytic solution with R(t_sf) = R_sf
output[:M]     = @. 4π/3 * μ * n_uniform * (output[:r] * pc)^3 / Msol
output[:v]     = p_sf_SN ./ output[:M]
output[:dA]    = 4π * output[:r].^2
output[:V]     = 4π/3 * output[:r].^3
output[:p]     = p_sf_SN
output[:f_kin] = @. 0.5 * (p_sf_SN * Msol * km_s)^2 / (output[:M] * Msol) / 1e51

Uniform_Medium["SN"] = output

In [ ]:
# Superbubble
n_uniform = Structured_Medium["SB"][:n_H] # ambient density measured at each time

# characteristic scales (corresponding to these densities)
t_sf_SB = 4.4e-2  * n_uniform.^-0.55
R_sf_SB = 22.6    * n_uniform.^-0.42
p_sf_SB = 3.165e5 * n_uniform.^-0.13

# number of SNe as a function of time
Num_SN = input2func(N_SN_SB_continuous)
E_SN   = 1e51 * input2func(SB_cont.E_51)
pdot_SB = 4π * α_p * p_ED / Δt_SB
p_SB   = p_sf_SB + pdot_SB * t_fil

# store output
output = Dict{Symbol, Vector{Float64}}()
output[:t]     = t_fil
output[:r]     = @. (R_sf_SB^4 + 3 / π * p_sf_SB * Msol * km_s / μ / n_uniform * max(t_fil - t_sf_SB, 0) * Myr * (1 + 0.5 * pdot_SB * (t_fil + t_sf_SB) / p_sf_SB) / pc^4)^0.25 # Analytic solution with R(t_sf) = R_sf
output[:M]     = @. 4π/3 * μ * n_uniform * (output[:r] * pc)^3 / Msol
output[:v]     = p_SB ./ output[:M]
output[:dA]    = 4π * output[:r].^2
output[:V]     = 4π/3 * output[:r].^3
output[:p]     = p_SB ./ Num_SN.(t_fil)
output[:f_kin] = @. 0.5 * (p_SB * Msol * km_s)^2 / (output[:M] * Msol) / E_SN(t_fil)

Uniform_Medium["SB"] = output

##### Plot model results

In [ ]:
# some useful definitions
model_names = ["SN", "SB"]
names = Dict("SN" => "Single SN", "SB" => L"\mathrm{Superbubble \ (\Delta t_{SN} = 1 \ Myr)}")
c_model = Dict("SN" => "gray", "SB" => "royalblue")

###### Globals Plot

Below is the corresponding "global plot".   
This showcases that at fixed time the blastwave expanding into the structured medium is **larger** than a blastwave in a uniform medium, with the same average density.   
This is largely driven by the fact that disproportionally more mass comes from the directions with dense structures, dominating the average density, despite most of the volume corresponding to low density medium.   
Curiously, the mass-weighted expansion velocity of the blastwave is below the uniform-medium expansion speed.    
As opposed to the mass, the momentum is largely independent of density, thus the mass-weighted speed is dominated by the denser directions.   

In [ ]:
p_sf_background = p_sf * n_0_background^-0.13 * 4π

# axis limits
tlim = [1e-1, 30]
rlim = [45.0, 400]
vlim = [0.1σ_fil, 200]
mlim = [3e3, 2e6]
plim = [0.5 * p_ED * 4π, 2.5 * p_sf_background]
Elim = [1.5e-4, 0.9];
nlim = [0.9 * n_0_background, 2 * n_fil]

In [ ]:
# create the figure and axes to plot the results
fig, axs = create_figure_filaments(t_fil, Structured_Medium, model_names, names, c_model, tlim)
plot_reference_models!(axs, Uniform_Medium, model_names, c_model, update_legend=false)

# add y-axis limits
axs[1,1].set_ylim(rlim...)
axs[1,2].set_ylim(vlim...)
axs[1,3].set_ylim(mlim...)
axs[2,1].set_ylim(plim...)
axs[2,2].set_ylim(Elim...)
axs[2,3].set_ylim(nlim...)

# Speed
add_horizontal_annotation(axs[1,2], σ_fil, label=L"\mathrm{σ_{fil}}", xlabel=5e-2)

# Momentum
add_horizontal_annotation(axs[2,1], p_sf_background, label=L"\mathrm{p_{sf}}", xlabel=5e-2)
add_horizontal_annotation(axs[2,1], 4π * α_p * p_ED, label=L"\mathrm{\alpha_{p} \times p_{SN}}", xlabel=5e-2)

# Density
add_horizontal_annotation(axs[2,3], n_0_background, label=L"\mathrm{n_{0}}", xlabel=5e-2)
add_horizontal_annotation(axs[2,3], n_fil, label=L"\mathrm{n_{fil}}", xlabel=5e-2)

##### Expansion speed

Evidently, the mass-weighted speed does not reliably capture the expansion of the remnant.   
It is thus instructive to inspect the rate at which the volume changes to construct an average expansion speed.   
As it turns out, this expansion speed matches almost perfectly with the instantaneous speed of the uniform-density model at the current average density.   
In other words: blastwaves have memory of the ambient medium they expanded through.   

An observer using the current density and size, will necessarily overestimate the age of a remnant (provided the remnant expanded through on average more diffuse gas than is measured)!

In [ ]:
# plot expansion speed
fig, ax = plot_expansion_speed_filaments(t_fil, Structured_Medium, Uniform_Medium, model_names, names, c_model)
# add y-axis limits
ax.set_xlim(tlim...)
ax.set_ylim(vlim...)

###### Geometry tracks

Here is the geometry track of the blastwaves in the structured medium.   
The structured medium can lead to more prolate and more oblate geometries depending on various aspects of the geometry and the explosion model.  

In [ ]:
# Create figure showing bubble shapes
fig, ax = plot_geometry_tracks_filaments(t_fil, Structured_Medium, model_names, names, c_model, [1, (5:5:30)...])